# 🚀 TITAN 2.0 - Live Real-Time Execution System (Google Colab)

**Production-Ready End-to-End Live Execution with Comprehensive Build & Monitoring**

This notebook provides a complete end-to-end system for live real-time arbitrage execution:
- ✅ Full automated build and dependency installation
- ✅ Live execution mode with real blockchain transactions
- ✅ Real-time execution journal (drum) for tracking all operations
- ✅ Comprehensive monitoring and health checks
- ✅ Safety checks and circuit breakers
- ✅ Performance metrics and profit tracking

---

## ⚠️ CRITICAL: Live Execution Warnings

1. **REAL MONEY**: This notebook executes REAL blockchain transactions with REAL funds
2. **GAS COSTS**: Every transaction consumes real ETH/MATIC/etc. for gas fees
3. **RISK OF LOSS**: You can lose money from failed transactions, gas costs, and market movements
4. **SECURITY**: Never share your private keys. Use a dedicated wallet with minimal funds.
5. **TESTING**: ALWAYS test in PAPER mode first before switching to LIVE

## 📋 Prerequisites

Before starting, ensure you have:
- [ ] Dedicated wallet with private key (NOT your main wallet)
- [ ] Gas funds on target networks (start with $50-100 worth)
- [ ] Infura or Alchemy API keys for RPC access
- [ ] Understanding of the risks involved
- [ ] Tested the system in PAPER mode first

---

## 🛠️ Step 1: Complete System Build

This cell performs a full automated build including:
- System dependencies (Node.js, Python packages, Redis)
- Repository cloning
- Dependency installation
- Build verification
- Environment setup

In [ ]:
%%bash
set -e  # Exit on error

echo "🔧 ===== TITAN 2.0 COMPLETE SYSTEM BUILD ====="
echo ""
echo "📦 Step 1/6: Installing System Dependencies..."

# Update package list
apt-get update -qq > /dev/null 2>&1

# Install Node.js 18.x
echo "   Installing Node.js 18.x..."
curl -fsSL https://deb.nodesource.com/setup_18.x | bash - > /dev/null 2>&1
apt-get install -y nodejs > /dev/null 2>&1

# Install Redis
echo "   Installing Redis server..."
apt-get install -y redis-server > /dev/null 2>&1

# Install build essentials
echo "   Installing build tools..."
apt-get install -y build-essential git curl wget python3-dev > /dev/null 2>&1

echo ""
echo "✅ System dependencies installed!"
echo "   Node.js: $(node --version)"
echo "   npm: $(npm --version)"
echo "   Python: $(python3 --version)"
echo "   Redis: $(redis-server --version | head -n1)"
echo ""

echo "📥 Step 2/6: Cloning TITAN Repository..."
cd /content
rm -rf Titan2.0
git clone https://github.com/vegas-max/Titan2.0.git > /dev/null 2>&1
echo "✅ Repository cloned to /content/Titan2.0"
echo ""

echo "📦 Step 3/6: Installing Python Dependencies..."
cd /content/Titan2.0
pip install -q -r requirements.txt
echo "✅ Python packages installed"
echo ""

echo "📦 Step 4/6: Installing Node.js Dependencies..."
npm install --silent --legacy-peer-deps
echo "✅ Node.js packages installed"
echo ""

echo "🔍 Step 5/6: Verifying Build..."
# Verify critical files exist
if [ -f "mainnet_orchestrator.py" ] && [ -f "offchain/execution/bot.js" ]; then
    echo "✅ Core files verified"
else
    echo "❌ Critical files missing!"
    exit 1
fi
echo ""

echo "📁 Step 6/6: Creating Required Directories..."
mkdir -p signals/outgoing signals/processed data/logs data/metrics
echo "✅ Directory structure created"
echo ""

echo "🎉 ===== BUILD COMPLETE ====="
echo ""
echo "✅ All systems ready for execution!"
echo "   Location: /content/Titan2.0"
echo ""

## 🔐 Step 2: Live Execution Configuration

**CRITICAL**: This configures the system for LIVE mode with REAL transactions.

⚠️ **START WITH MINIMAL FUNDS** - Test with small amounts first!

In [ ]:
import os
from getpass import getpass
import json

print("🔐 LIVE EXECUTION CONFIGURATION")
print("=" * 70)
print()
print("⚠️  WARNING: You are configuring LIVE MODE with REAL money")
print("⚠️  START SMALL: Use a test wallet with minimal funds ($50-100)")
print("⚠️  UNDERSTAND RISKS: You can lose money from failed trades and gas costs")
print()
confirm = input("Do you understand and accept these risks? Type 'YES' to continue: ").strip()

if confirm != 'YES':
    print("\n❌ Configuration cancelled. Please review the risks before proceeding.")
    raise SystemExit("User cancelled live configuration")

print("\n✅ Proceeding with live configuration...")
print()

# Wallet Configuration
print("🔑 WALLET CONFIGURATION")
print("-" * 70)
private_key = getpass("Enter your wallet private key (hidden): ").strip()
if not private_key:
    raise ValueError("Private key is required for live execution")

# Remove 0x prefix if present
if private_key.startswith('0x'):
    private_key = private_key[2:]

if len(private_key) != 64:
    raise ValueError(f"Invalid private key length: {len(private_key)}. Expected 64 hex characters.")

print("✅ Private key validated")
print()

# RPC Configuration
print("🌐 RPC PROVIDER CONFIGURATION")
print("-" * 70)
infura_id = input("Infura Project ID: ").strip()
alchemy_key = input("Alchemy API Key (optional, press Enter to skip): ").strip()
print()

# Target Networks
print("🎯 TARGET NETWORKS")
print("-" * 70)
print("Select networks for live execution (recommended: start with Polygon for low gas)")
print("Available networks:")
networks = {
    '1': 'Ethereum (high gas)',
    '137': 'Polygon (recommended - low gas)',
    '42161': 'Arbitrum (low gas)',
    '10': 'Optimism (low gas)',
    '8453': 'Base (low gas)'
}

for key, name in networks.items():
    print(f"  {key}: {name}")

selected_networks = input("\nEnter network IDs (comma-separated, e.g., 137,42161): ").strip()
if not selected_networks:
    selected_networks = '137'  # Default to Polygon
print(f"✅ Selected networks: {selected_networks}")
print()

# Safety Configuration
print("🛡️ SAFETY CONFIGURATION")
print("-" * 70)
print("Configure safety limits to protect your funds:")
print()

max_gas_gwei = input("Maximum gas price (gwei) [default: 100]: ").strip() or "100"
min_profit_usd = input("Minimum profit per trade (USD) [default: 5]: ").strip() or "5"
max_slippage_bps = input("Maximum slippage (basis points) [default: 50]: ").strip() or "50"
circuit_breaker_fails = input("Circuit breaker: failures before pause [default: 5]: ").strip() or "5"

print()
print("✅ Safety limits configured:")
print(f"   Max gas: {max_gas_gwei} gwei")
print(f"   Min profit: ${min_profit_usd}")
print(f"   Max slippage: {max_slippage_bps} bps ({float(max_slippage_bps)/100}%)")
print(f"   Circuit breaker: {circuit_breaker_fails} consecutive failures")
print()

# Build RPC URLs
rpc_ethereum = f"https://mainnet.infura.io/v3/{infura_id}" if infura_id else ""
rpc_polygon = f"https://polygon-mainnet.infura.io/v3/{infura_id}" if infura_id else ""
rpc_arbitrum = f"https://arbitrum-mainnet.infura.io/v3/{infura_id}" if infura_id else ""
rpc_optimism = f"https://optimism-mainnet.infura.io/v3/{infura_id}" if infura_id else ""
rpc_base = f"https://base-mainnet.infura.io/v3/{infura_id}" if infura_id else ""

# Create .env file
env_content = f"""# TITAN LIVE EXECUTION CONFIGURATION
# Generated: {os.popen('date').read().strip()}
# ⚠️  LIVE MODE - REAL MONEY - USE WITH CAUTION

# EXECUTION MODE
EXECUTION_MODE=LIVE
TITAN_EXECUTION_MODE=LIVE

# WALLET CONFIGURATION
PRIVATE_KEY={private_key}

# RPC PROVIDERS
INFURA_PROJECT_ID={infura_id}
ALCHEMY_API_KEY={alchemy_key}

RPC_ETHEREUM={rpc_ethereum}
RPC_POLYGON={rpc_polygon}
RPC_ARBITRUM={rpc_arbitrum}
RPC_OPTIMISM={rpc_optimism}
RPC_BASE={rpc_base}

# TARGET NETWORKS
TARGET_NETWORKS={selected_networks}

# SAFETY LIMITS
MAX_BASE_FEE_GWEI={max_gas_gwei}
MIN_PROFIT_USD={min_profit_usd}
MAX_SLIPPAGE_BPS={max_slippage_bps}
CIRCUIT_BREAKER_MAX_CONSECUTIVE_FAILURES={circuit_breaker_fails}

# FLASH LOAN CONFIGURATION (Zero capital required)
FLASH_LOAN_ENABLED=true
FLASH_LOAN_PROVIDER=1

# SIMULATION (CRITICAL for live mode)
ENFORCE_SIMULATION=true

# FILL SYSTEM (All-or-nothing for safety)
COWSWAP_PARTIALLY_FILLABLE=false

# FEATURES
ENABLE_CROSS_CHAIN=false
ENABLE_MEV_PROTECTION=true
ENABLE_REALTIME_TRAINING=true

# MONITORING
LOG_LEVEL=INFO
REDIS_URL=redis://localhost:6379
"""

# Write configuration
with open('/content/Titan2.0/.env', 'w') as f:
    f.write(env_content)

print("💾 Configuration saved to .env")
print()
print("=" * 70)
print("✅ LIVE EXECUTION CONFIGURED")
print("=" * 70)
print()
print("⚠️  REMEMBER:")
print("   1. This will execute REAL transactions")
print("   2. You will spend REAL gas fees")
print("   3. You can LOSE money")
print("   4. Monitor carefully and be ready to stop")
print("   5. Start with SMALL amounts")
print()
print(f"📊 Your safety limits:")
print(f"   • Max gas: {max_gas_gwei} gwei")
print(f"   • Min profit: ${min_profit_usd}")
print(f"   • Max slippage: {float(max_slippage_bps)/100}%")
print(f"   • Circuit breaker: {circuit_breaker_fails} failures")
print()
print("🔒 Private key: " + "*" * 60 + private_key[-4:])
print()

## 🚀 Step 3: Start Redis and Services

Start all required background services.

In [ ]:
%%bash
echo "🚀 Starting Background Services..."
echo ""

# Start Redis
echo "📊 Starting Redis..."
redis-server --daemonize yes --port 6379
sleep 2

if redis-cli ping > /dev/null 2>&1; then
    echo "✅ Redis is running (port 6379)"
else
    echo "⚠️  Redis failed to start (will use file-based fallback)"
fi

echo ""
echo "✅ Services ready!"
echo ""

## 📓 Step 4: Initialize Execution Journal (Drum)

Create a comprehensive tracking system for all executions.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

# Create execution journal directory
journal_dir = Path('/content/Titan2.0/data/execution_journal')
journal_dir.mkdir(parents=True, exist_ok=True)

# Initialize journal metadata
journal_meta = {
    "session_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "started_at": datetime.now().isoformat(),
    "mode": "LIVE",
    "executions": [],
    "statistics": {
        "total_attempts": 0,
        "successful": 0,
        "failed": 0,
        "total_profit_usd": 0.0,
        "total_gas_spent_usd": 0.0,
        "net_profit_usd": 0.0
    },
    "safety_events": [],
    "circuit_breaker": {
        "active": False,
        "consecutive_failures": 0,
        "triggered_at": None
    }
}

journal_file = journal_dir / f"journal_{journal_meta['session_id']}.json"
with open(journal_file, 'w') as f:
    json.dump(journal_meta, f, indent=2)

print("📓 EXECUTION JOURNAL INITIALIZED")
print("=" * 70)
print(f"Session ID: {journal_meta['session_id']}")
print(f"Started: {journal_meta['started_at']}")
print(f"Mode: {journal_meta['mode']}")
print(f"Journal file: {journal_file}")
print()
print("✅ Ready to track all executions!")
print()

# Export for use in other cells
%store journal_file
%store journal_meta

## 🧠 Step 5: Start TITAN Brain (Intelligence Layer)

Launch the Brain for opportunity detection and analysis.

In [ ]:
import subprocess
import threading
import time
import os

def start_brain():
    """Start the TITAN Brain in background"""
    os.chdir('/content/Titan2.0')
    # Set execution mode explicitly
    env = os.environ.copy()
    env['EXECUTION_MODE'] = 'LIVE'
    env['TITAN_EXECUTION_MODE'] = 'LIVE'
    
    subprocess.Popen(
        ['python3', 'mainnet_orchestrator.py'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

print("🧠 Starting TITAN Brain (Mainnet Orchestrator)...")
print("=" * 70)
print()

# Start brain in background thread
brain_thread = threading.Thread(target=start_brain, daemon=True)
brain_thread.start()

# Give it time to initialize
print("⏳ Initializing Brain components...")
time.sleep(10)

print()
print("✅ TITAN Brain is running!")
print()
print("Brain is now:")
print("   • Scanning blockchain networks for opportunities")
print("   • Running AI/ML models for prediction")
print("   • Analyzing arbitrage possibilities")
print("   • Publishing trade signals for execution")
print()
print("📊 Mode: LIVE (Real blockchain data → Real calculations)")
print()

## 🤖 Step 6: Start TITAN Bot (Execution Layer)

**CRITICAL**: This starts LIVE execution with REAL transactions!

In [ ]:
import subprocess
import threading
import time
import os

def start_bot():
    """Start the TITAN Bot for live execution"""
    os.chdir('/content/Titan2.0')
    # Set execution mode explicitly
    env = os.environ.copy()
    env['EXECUTION_MODE'] = 'LIVE'
    env['TITAN_EXECUTION_MODE'] = 'LIVE'
    
    subprocess.Popen(
        ['node', 'offchain/execution/bot.js'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

print("🤖 Starting TITAN Bot (Live Execution Layer)...")
print("=" * 70)
print()
print("⚠️  ⚠️  ⚠️  LIVE MODE ACTIVE ⚠️  ⚠️  ⚠️")
print()
print("The bot will now execute REAL transactions on the blockchain!")
print("This means:")
print("   • Real gas fees will be spent")
print("   • Real funds will be used for flash loans")
print("   • Real profits or losses will occur")
print()
print("⏳ Initializing Bot components...")

# Start bot in background thread
bot_thread = threading.Thread(target=start_bot, daemon=True)
bot_thread.start()

# Give it time to initialize
time.sleep(10)

print()
print("✅ TITAN Bot is running in LIVE MODE!")
print()
print("Bot is now:")
print("   • Listening for trade signals from Brain")
print("   • Simulating transactions before execution")
print("   • Executing REAL transactions on blockchain")
print("   • Monitoring gas prices and network conditions")
print("   • Enforcing safety limits and circuit breakers")
print()
print("🚨 MONITOR CAREFULLY - REAL MONEY AT STAKE!")
print()

## 📊 Step 7: Real-Time Execution Monitor

Monitor live executions, profits, and system health in real-time.

In [ ]:
import json
import time
import os
from pathlib import Path
from datetime import datetime
from IPython.display import clear_output, display, HTML

def load_journal():
    """Load the latest execution journal"""
    try:
        %store -r journal_file
        with open(journal_file, 'r') as f:
            return json.load(f)
    except:
        return None

def scan_signals():
    """Scan for new signals and executions"""
    signals_dir = Path('/content/Titan2.0/signals/outgoing')
    processed_dir = Path('/content/Titan2.0/signals/processed')
    
    pending = len(list(signals_dir.glob('*.json'))) if signals_dir.exists() else 0
    processed = len(list(processed_dir.glob('*.json'))) if processed_dir.exists() else 0
    
    return pending, processed

def format_dashboard():
    """Generate real-time dashboard HTML"""
    journal = load_journal()
    pending, processed = scan_signals()
    
    if journal:
        stats = journal.get('statistics', {})
        circuit = journal.get('circuit_breaker', {})
    else:
        stats = {'total_attempts': 0, 'successful': 0, 'failed': 0, 
                'total_profit_usd': 0, 'total_gas_spent_usd': 0, 'net_profit_usd': 0}
        circuit = {'active': False, 'consecutive_failures': 0}
    
    success_rate = (stats['successful'] / stats['total_attempts'] * 100) if stats['total_attempts'] > 0 else 0
    
    html = f"""
    <div style="font-family: monospace; background: #1a1a1a; color: #00ff00; padding: 20px; border-radius: 10px;">
        <h2 style="color: #00ff00; margin-top: 0;">🚀 TITAN LIVE EXECUTION MONITOR</h2>
        <p style="color: #ff6b6b; font-weight: bold;">⚠️  LIVE MODE - REAL TRANSACTIONS ACTIVE</p>
        <hr style="border-color: #00ff00;">
        
        <h3 style="color: #4ecdc4;">📊 Execution Statistics</h3>
        <table style="width: 100%; color: #00ff00;">
            <tr><td>Total Attempts:</td><td style="text-align: right; font-weight: bold;">{stats['total_attempts']}</td></tr>
            <tr><td>✅ Successful:</td><td style="text-align: right; color: #00ff00; font-weight: bold;">{stats['successful']}</td></tr>
            <tr><td>❌ Failed:</td><td style="text-align: right; color: #ff6b6b; font-weight: bold;">{stats['failed']}</td></tr>
            <tr><td>Success Rate:</td><td style="text-align: right; font-weight: bold;">{success_rate:.1f}%</td></tr>
        </table>
        
        <h3 style="color: #4ecdc4;">💰 Profit & Loss</h3>
        <table style="width: 100%; color: #00ff00;">
            <tr><td>Gross Profit:</td><td style="text-align: right; color: #00ff00; font-weight: bold;">${stats['total_profit_usd']:.2f}</td></tr>
            <tr><td>Gas Spent:</td><td style="text-align: right; color: #ff6b6b; font-weight: bold;">-${stats['total_gas_spent_usd']:.2f}</td></tr>
            <tr style="border-top: 1px solid #00ff00;"><td><strong>Net Profit:</strong></td><td style="text-align: right; color: {'#00ff00' if stats['net_profit_usd'] >= 0 else '#ff6b6b'}; font-weight: bold; font-size: 1.2em;">${stats['net_profit_usd']:.2f}</td></tr>
        </table>
        
        <h3 style="color: #4ecdc4;">📡 Signal Queue</h3>
        <table style="width: 100%; color: #00ff00;">
            <tr><td>Pending Signals:</td><td style="text-align: right; font-weight: bold;">{pending}</td></tr>
            <tr><td>Processed Signals:</td><td style="text-align: right; font-weight: bold;">{processed}</td></tr>
        </table>
        
        <h3 style="color: #4ecdc4;">🛡️ Circuit Breaker</h3>
        <table style="width: 100%; color: #00ff00;">
            <tr><td>Status:</td><td style="text-align: right; color: {'#ff6b6b' if circuit['active'] else '#00ff00'}; font-weight: bold;">{'🔴 ACTIVE (System Paused)' if circuit['active'] else '🟢 Normal'}</td></tr>
            <tr><td>Consecutive Failures:</td><td style="text-align: right; color: {'#ff6b6b' if circuit['consecutive_failures'] >= 3 else '#00ff00'}; font-weight: bold;">{circuit['consecutive_failures']}</td></tr>
        </table>
        
        <p style="color: #888; font-size: 0.8em; margin-top: 20px;">Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    </div>
    """
    return html

# Real-time monitoring loop
print("📊 Starting real-time monitor...")
print("   (Run this cell to see live updates)")
print()
print("💡 TIP: Stop this cell to pause monitoring, system continues running")
print()

try:
    while True:
        clear_output(wait=True)
        display(HTML(format_dashboard()))
        time.sleep(5)  # Update every 5 seconds
except KeyboardInterrupt:
    print("\n⏸️  Monitoring paused (system still running)")

## 📈 Step 8: View Execution History

View detailed history of all executions.

In [ ]:
import json
import pandas as pd
from pathlib import Path

def load_execution_history():
    """Load and display execution history"""
    try:
        %store -r journal_file
        with open(journal_file, 'r') as f:
            journal = json.load(f)
        
        executions = journal.get('executions', [])
        
        if not executions:
            print("📭 No executions yet")
            return
        
        # Convert to DataFrame for nice display
        df = pd.DataFrame(executions)
        
        print("📊 EXECUTION HISTORY")
        print("=" * 100)
        print()
        print(df.to_string())
        print()
        print(f"Total executions: {len(executions)}")
        
        # Summary stats
        if 'profit_usd' in df.columns:
            print()
            print("Summary:")
            print(f"  Average profit: ${df['profit_usd'].mean():.2f}")
            print(f"  Best trade: ${df['profit_usd'].max():.2f}")
            print(f"  Worst trade: ${df['profit_usd'].min():.2f}")
        
    except Exception as e:
        print(f"❌ Error loading history: {e}")

load_execution_history()

## 📋 Step 9: Check System Health

Verify all components are running correctly.

In [ ]:
%%bash
echo "🏥 SYSTEM HEALTH CHECK"
echo "=" | head -c 70
echo ""
echo ""

echo "📊 Redis Status:"
if redis-cli ping > /dev/null 2>&1; then
    echo "   ✅ Redis is running"
    echo "   Keys: $(redis-cli dbsize | grep -o '[0-9]*')"
else
    echo "   ❌ Redis is not running"
fi
echo ""

echo "🧠 Brain (Python) Status:"
if pgrep -f "mainnet_orchestrator.py" > /dev/null; then
    echo "   ✅ Brain is running (PID: $(pgrep -f mainnet_orchestrator.py))"
else
    echo "   ❌ Brain is not running"
fi
echo ""

echo "🤖 Bot (Node.js) Status:"
if pgrep -f "bot.js" > /dev/null; then
    echo "   ✅ Bot is running (PID: $(pgrep -f bot.js))"
else
    echo "   ❌ Bot is not running"
fi
echo ""

echo "📁 Signal Files:"
cd /content/Titan2.0
if [ -d "signals/outgoing" ]; then
    pending=$(find signals/outgoing -name '*.json' 2>/dev/null | wc -l)
    echo "   Pending: $pending"
else
    echo "   Pending: 0"
fi
if [ -d "signals/processed" ]; then
    processed=$(find signals/processed -name '*.json' 2>/dev/null | wc -l)
    echo "   Processed: $processed"
else
    echo "   Processed: 0"
fi
echo ""

echo "✅ Health check complete!"
echo ""

## 🛑 Step 10: Emergency Stop

**USE THIS TO STOP ALL LIVE EXECUTION IMMEDIATELY**

In [ ]:
%%bash
echo "🛑 ===== EMERGENCY STOP ====="
echo ""
echo "⚠️  Stopping all TITAN components..."
echo ""

# Stop Python processes (Brain)
echo "🧠 Stopping Brain..."
pkill -f "mainnet_orchestrator.py" 2>/dev/null || true
echo "   ✅ Brain stopped"

# Stop Node.js processes (Bot)
echo "🤖 Stopping Bot..."
pkill -f "bot.js" 2>/dev/null || true
echo "   ✅ Bot stopped"

# Stop Redis
echo "📊 Stopping Redis..."
redis-cli shutdown 2>/dev/null || true
echo "   ✅ Redis stopped"

echo ""
echo "✅ All systems stopped!"
echo ""
echo "💾 Execution journal preserved in /content/Titan2.0/data/execution_journal/"
echo ""

## 📊 Step 11: Generate Performance Report

Generate a comprehensive performance report.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

def generate_report():
    """Generate comprehensive performance report"""
    try:
        %store -r journal_file
        with open(journal_file, 'r') as f:
            journal = json.load(f)
        
        stats = journal.get('statistics', {})
        
        report = f"""
═══════════════════════════════════════════════════════════════════
              TITAN 2.0 LIVE EXECUTION REPORT
═══════════════════════════════════════════════════════════════════

Session Information:
───────────────────────────────────────────────────────────────────
Session ID:     {journal['session_id']}
Started:        {journal['started_at']}
Mode:           {journal['mode']}
Duration:       {(datetime.now() - datetime.fromisoformat(journal['started_at'])).total_seconds() / 3600:.2f} hours

Execution Statistics:
───────────────────────────────────────────────────────────────────
Total Attempts:      {stats['total_attempts']}
Successful:          {stats['successful']} ({stats['successful']/stats['total_attempts']*100 if stats['total_attempts'] > 0 else 0:.1f}%)
Failed:              {stats['failed']} ({stats['failed']/stats['total_attempts']*100 if stats['total_attempts'] > 0 else 0:.1f}%)

Financial Performance:
───────────────────────────────────────────────────────────────────
Gross Profit:        ${stats['total_profit_usd']:.2f}
Gas Costs:           ${stats['total_gas_spent_usd']:.2f}
NET PROFIT:          ${stats['net_profit_usd']:.2f}

Average per Trade:   ${stats['net_profit_usd']/stats['successful'] if stats['successful'] > 0 else 0:.2f}

Safety Events:
───────────────────────────────────────────────────────────────────
Circuit Breaker Triggers: {len(journal.get('safety_events', []))}
Current Status:           {'🔴 ACTIVE' if journal.get('circuit_breaker', {}).get('active') else '🟢 Normal'}

═══════════════════════════════════════════════════════════════════
Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
═══════════════════════════════════════════════════════════════════
"""
        print(report)
        
        # Save report to file
        report_file = Path(f'/content/Titan2.0/data/execution_journal/report_{journal["session_id"]}.txt')
        with open(report_file, 'w') as f:
            f.write(report)
        
        print(f"\n💾 Report saved to: {report_file}")
        
    except Exception as e:
        print(f"❌ Error generating report: {e}")

generate_report()

## 📚 Additional Resources

### Documentation
- **Main README**: `/content/Titan2.0/README.md`
- **Google Colab Guide**: `/content/Titan2.0/GOOGLE_COLAB_STEP_BY_STEP.md`
- **Operations Guide**: `/content/Titan2.0/OPERATIONS_GUIDE.md`
- **Security Summary**: `/content/Titan2.0/SECURITY_SUMMARY.md`

### Important Files
- **Configuration**: `/content/Titan2.0/.env`
- **Execution Journal**: `/content/Titan2.0/data/execution_journal/`
- **Logs**: `/content/Titan2.0/data/logs/`
- **Signals**: `/content/Titan2.0/signals/`

### Support
- **GitHub Repository**: https://github.com/vegas-max/Titan2.0
- **Issues**: https://github.com/vegas-max/Titan2.0/issues

---

## ⚠️ Final Disclaimer

**This is experimental software for live trading with real money:**

- ❌ **NOT FINANCIAL ADVICE**: This is for educational purposes only
- ❌ **RISK OF LOSS**: You can lose money from failed trades, gas costs, and market movements
- ❌ **NO WARRANTY**: Provided "as is" without any guarantees
- ❌ **YOUR RESPONSIBILITY**: You are responsible for all trades and their consequences

**Best Practices:**
- ✅ Always test in PAPER mode first
- ✅ Start with minimal funds
- ✅ Use a dedicated wallet (not your main wallet)
- ✅ Monitor continuously
- ✅ Be ready to stop immediately if needed
- ✅ Understand the risks before starting

**By using this notebook, you acknowledge and accept all risks.**

---

**Built with ❤️ by the Titan Team**

⭐ Star the repo if you find it useful!